#División de cédulas


In [ ]:
# Importar librerías

In [ ]:
import spacy
nlp=spacy.load("es_core_news_md")
from pathlib import Path
import re
import os
from collections import Counter
import csv

In [ ]:
# Leer el archivo

In [ ]:
with open("utblac_wbs_0496_pt1.txt", "r", encoding="utf-8") as f:
  texto = f.read()

In [ ]:
# Dividir cédulas usando [FIN] como divisor

In [ ]:
cedulas = re.findall(r'.*?\[FIN\]', texto, re.DOTALL)

In [ ]:
# Crear la carpeta para guardar los resultados

In [ ]:
os.makedirs("cedulas", exist_ok=True)

In [ ]:
# Guardar cada cédula como un archivo .txt

In [ ]:
for i, c in enumerate(cedulas):
  with open(f"cedulas/cedula_{i+1}.txt", "w", encoding="utf-8") as out:
    out.write(c.strip())

# Tokenización

In [ ]:
# Creación de directorio para input y output

In [ ]:
input_dir = Path("cedulas_1")
output_dir = Path("tokens_1")
output_dir.mkdir(exist_ok=True)

In [ ]:
# Función anidada para procesamiento del texto

In [ ]:
for file_path in input_dir.glob("*.txt"):
  with open(file_path, "r", encoding="utf-8") as f:
    text=f.read()
  doc=nlp(text)
  tokens = [token.text for token in doc if not token.is_space]
  output_file = output_dir / f"{file_path.stem}_tokens.txt"
  with open(output_file, "w", encoding="utf-8") as f:
    f.write("\n".join(tokens))
  print(f"Procesado: {file_path.name}")

# Reconocimiento de nombres y entidades (NER)


In [ ]:
# Establecer directorios de input y oputput:

In [ ]:
input_dir = Path("tokens_1")
output_dir = Path("NER_1")
output_dir.mkdir(exist_ok=True)

In [ ]:
# leer archivos con codificación utf-8
for file_path in input_dir.glob("*.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
# Pre-procesamiento de archivos archivos (identifiación)
    doc = nlp(text)
# imprimir entidades
    for ent in doc.ents:
        print(ent.text, ent.label_)
# guardar resultado en la carpeta seleccionada con output_dir con codificación utf-8
    output_file = output_dir / file_path.name
    with open(output_file, "w", encoding="utf-8") as out:
        for ent in doc.ents:
            out.write(f"{ent.text}\t{ent.label_}\n")

# Visualización de nombres y entidades

In [ ]:
# Visualización de entidades

In [ ]:
entidades = set()
for file_path in input_dir.glob("*.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    doc = nlp(text)
    for ent in doc.ents:
        entidades.add((ent.text, ent.label_))
for ent, label in sorted(entidades):
    print(ent, label)

In [ ]:
# Conteo de frecuencias por entidad

In [ ]:
counter = Counter()
for file_path in input_dir.glob("*.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    doc = nlp(text)
    for ent in doc.ents:
        key = (ent.text, ent.label_)
        counter[key] += 1

In [ ]:
# Visualización de conteo de frecuencia por entidad ordenados

In [ ]:
for (ent, label), freq in counter.most_common():
    print(f"{ent}\t{label}\t{freq}")

# Normalización de nombres y entidades

In [ ]:
# Definir función para ormalización básica

In [ ]:
def normalize(text):
    text = text.lower()                 # minúsculas
    text = text.strip()                 # espacios
    text = re.sub(r"\s+", " ", text)   # espacios múltiples
    text = re.sub(r"[^\w\sáéíóúñü]", "", text)  # quitar puntuación
    return text

In [ ]:
# Aplicar función para normalización y visualización del conteo de entidades

In [ ]:
counter = Counter()
for file_path in input_dir.glob("*.txt"):
    text = file_path.read_text(encoding="utf-8")
    doc = nlp(text)
    for ent in doc.ents:
        norm = normalize(ent.text)
        key = (norm, ent.label_)
        counter[key] += 1
for (ent, label), freq in counter.most_common():
    print(ent, label, freq)

In [ ]:
# Importar resultados para normalizarción en google sheets con validación de datos

In [ ]:
output_dir = Path("tokens_normalizados")
output_dir.mkdir(exist_ok=True)

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["entidad", "tipo", "frecuencia"])
    for (ent, label), freq in counter.most_common():
        writer.writerow([ent, label, freq])